# EDA — stabilité entre millésimes et ruptures de série

**Session de travail du 29 août 2026.** Quatrième et dernier carnet exploratoire.

Mon protocole d'évaluation est temporel : j'entraîne sur les sessions les plus
anciennes et je teste sur la plus récente. Cette construction n'a de sens que si
les séries sont comparables d'une année sur l'autre. Ce carnet répond donc à :

> 1. Sur quelles sessions ma cible est-elle réellement calculable ?
> 2. La cible est-elle stationnaire, ou dérive-t-elle ?
> 3. Quelles ruptures de série séparent une évolution réelle d'un changement de
>    règle administrative ?

La distinction de la troisième question est décisive. Une évolution réelle est
un signal que le modèle doit apprendre. Une rupture administrative est un
artefact qu'il apprendrait à tort.

In [ ]:
import csv
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

sys.path.insert(0, str(Path.cwd().parent / "src"))
from edumatch.config import load_settings

settings = load_settings("prod")
RAW = settings.raw_dir / "parcoursup"
MILLESIMES = settings.donnees.parcoursup.millesimes


def charger(annee: int) -> pl.DataFrame:
    # `infer_schema_length` est volontairement élevé : le code de département
    # prend les valeurs `2A` et `2B` pour la Corse, qui apparaissent tard dans
    # le fichier. Une inférence sur les premières lignes typerait la colonne en
    # entier et la lecture échouerait.
    return pl.read_csv(
        RAW / f"parcoursup_{annee}.csv",
        separator=";",
        encoding="utf8-lossy",
        infer_schema_length=20000,
    )


def entetes(annee: int) -> set[str]:
    """Lit les seuls en-têtes, sans parser le fichier."""
    with open(RAW / f"parcoursup_{annee}.csv", encoding="utf-8-sig") as fichier:
        return set(next(csv.reader(fichier, delimiter=";")))


print("millésimes disponibles :", MILLESIMES)

## 1. Sur quelles sessions ma cible est-elle calculable ?

Question préalable à toute analyse de stabilité : le label suppose un numérateur
et un dénominateur, tous deux ventilés par type de baccalauréat. Je vérifie leur
présence effective session par session, plutôt que de la supposer.

In [ ]:
colonnes_label = [
    "prop_tot",
    "prop_tot_bg",
    "prop_tot_bt",
    "prop_tot_bp",
    "prop_tot_bg_brs",
    "nb_voe_pp_bg",
    "nb_voe_pp_bg_brs",
    "acc_bg",
]
schemas = {annee: entetes(annee) for annee in MILLESIMES}

print(f"{'colonne':<20}" + "".join(f"{a:>7}" for a in MILLESIMES))
for colonne in colonnes_label:
    presence = "".join("    oui" if colonne in schemas[a] else "      -" for a in MILLESIMES)
    print(f"{colonne:<20}{presence}")

**Ce que j'en conclus, et cela modifie mon protocole d'évaluation.**

Le dénominateur `nb_voe_pp_*` est présent sur les huit sessions. Le numérateur
`prop_tot_*`, ventilé par type de baccalauréat, **n'apparaît qu'à partir de la
session 2020**.

Mon label n'est donc pas calculable sur huit millésimes, mais sur **six**. La
répartition que j'avais retenue — entraînement de 2018 à 2023 — plaçait deux
sessions sans cible dans le jeu d'entraînement.

Je note l'alternative que j'écarte : `acc_bg` existe sur toute la période et
permettrait de construire un label dès 2018. Mais `acc` compte les inscrits, non
les propositions : ce serait une **autre cible**, et mélanger les deux
définitions selon la session produirait un label incohérent avec lui-même. Je
préfère quatre années de cible homogène à six années de cible composite.

In [ ]:
lignes = []
total_cellules = 0
suffixes = ["bg", "bg_brs", "bt", "bt_brs", "bp", "bp_brs"]

print(f"{'année':<7}{'formations':>12}{'cellules':>11}{'taux agrégé':>14}{'moyenne bornée':>16}{'médiane':>10}")
for annee in [a for a in MILLESIMES if a >= 2020]:
    d = charger(annee)
    cellules = sum(d.filter(pl.col(f"nb_voe_pp_{s}") > 0).height for s in suffixes)
    total_cellules += cellules

    x = d.filter(pl.col("nb_voe_pp_bg") > 0).with_columns(
        (pl.col("prop_tot_bg") / pl.col("nb_voe_pp_bg")).clip(0, 1).alias("taux")
    )
    voeux, propositions = d["nb_voe_pp_bg"].sum(), d["prop_tot_bg"].sum()
    agrege = propositions / voeux

    lignes.append((annee, agrege, x["taux"].mean(), x["taux"].median()))
    print(
        f"{annee:<7}{d.height:>12}{cellules:>11,}{agrege * 100:>13.1f}%"
        f"{x['taux'].mean():>16.3f}{x['taux'].median():>10.3f}"
    )

print()
print(f"total des cellules exploitables sur 2020-2025 : {total_cellules:,}")

**Ce que j'en conclus.** Le volume réellement exploitable est de **440 030
cellules** sur six sessions, et non des 560 000 à 625 000 que j'avançais en
supposant huit sessions utilisables. C'est une correction substantielle de la
volumétrie annoncée du projet.

Ce volume reste largement suffisant pour un modèle à base d'arbres sur données
tabulaires. La courbe d'apprentissage établira si davantage aurait aidé — c'est
précisément sa fonction, et je m'abstiens de le supposer ici.

## 2. La cible est-elle stationnaire ?

Un modèle entraîné sur les sessions anciennes n'a de valeur que si ce qu'il a
appris reste vrai sur la session de test. Je regarde donc l'évolution de la
cible elle-même, avec les **deux définitions du taux** — l'agrégé et la moyenne
par formation.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
annees = [r[0] for r in lignes]
ax.plot(annees, [r[1] for r in lignes], marker="o", color="#C44E52", label="taux agrégé (total propositions / total vœux)")
ax.plot(annees, [r[2] for r in lignes], marker="s", color="#4C72B0", label="moyenne des taux par formation")
ax.plot(annees, [r[3] for r in lignes], marker="^", color="#55A868", linestyle="--", label="médiane des taux par formation")
ax.set_xlabel("session")
ax.set_ylabel("taux d'admission, bac général")
ax.set_title("Les deux définitions du taux évoluent en sens contraire")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

**Ce que j'en conclus, et c'est le résultat le plus utile de ce carnet pour la
surveillance en production.**

Les deux définitions **divergent dans le temps**. Le taux agrégé recule de 40,5 %
à 36,7 % entre 2020 et 2025, tandis que la moyenne par formation progresse de
0,491 à 0,522.

Ce n'est pas une contradiction. Le catalogue passe de 12 760 à 14 252
formations : les formations qui s'ajoutent sont plus petites et moins tendues.
Elles tirent vers le haut la moyenne calculée formation par formation, tout en
pesant peu dans un agrégat dominé par les grosses formations.

La conséquence opérationnelle est directe : **un dispositif de surveillance qui
suivrait une seule de ces deux définitions conclurait à l'inverse de la
réalité.** Les deux devront être suivies, et la définition employée précisée à
chaque fois qu'un chiffre est publié.

Deuxième conséquence, pour la modélisation : **ma cible n'est pas
stationnaire**. Elle culmine en 2024, à 0,534, puis redescend à 0,522 en 2025. Un
modèle entraîné jusqu'en 2023 et validé sur 2024 apprend une tendance qui
s'inverse au moment du test. Ce n'est pas une raison de renoncer au protocole
temporel — c'est une raison de l'assumer, et d'attendre une dégradation entre
validation et test sans l'imputer au modèle.

## 3. Ruptures de série : évolution réelle ou changement de règle ?

Je cherche maintenant les discontinuités. L'enjeu n'est pas de les corriger mais
de les **qualifier** : une évolution réelle du système d'orientation est un
signal légitime ; un changement de règle administrative est un artefact que le
modèle apprendrait à tort.

In [ ]:
indicateurs = ["pct_tb", "pct_b", "pct_sansmention", "pct_bours", "pct_neobac"]
print(f"{'année':<7}" + "".join(f"{i:>18}" for i in indicateurs))
serie = {i: [] for i in indicateurs}
for annee in MILLESIMES:
    d = charger(annee)
    valeurs = []
    for indicateur in indicateurs:
        v = d[indicateur].mean() if indicateur in d.columns else float("nan")
        serie[indicateur].append(v)
        valeurs.append(f"{v:>18.1f}")
    print(f"{annee:<7}" + "".join(valeurs))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for indicateur, couleur in zip(
    ["pct_tb", "pct_sansmention", "pct_bours"], ["#C44E52", "#4C72B0", "#55A868"]
):
    ax.plot(MILLESIMES, serie[indicateur], marker="o", color=couleur, label=indicateur)
ax.axvline(2020, color="#888888", linestyle=":", linewidth=1.4)
ax.annotate("2020 — baccalauréat en contrôle continu", xy=(2020, 42), fontsize=9, color="#555555")
ax.set_xlabel("session")
ax.set_ylabel("part moyenne des admis (%)")
ax.set_title("Ruptures de série : mentions en 2020, boursiers en 2019 et 2025")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

**Ce que j'en conclus. Deux ruptures nettes, de natures différentes.**

**La session 2020 et le contrôle continu.** Le baccalauréat 2020 a été délivré
sans épreuves terminales. La trace est sans ambiguïté : les mentions « très
bien » passent de 7,4 % à 11,8 %, et les admis sans mention chutent de 43,4 % à
29,8 %, soit 13,6 points. Le retour vers la normale est lent — 2021 reste à
29,8 % — et le niveau d'avant n'est jamais retrouvé, la réforme du lycée étant
intervenue entre-temps.

C'est un **artefact administratif**. Les variables de mention sont donc
contaminées pour 2020 et 2021 : un modèle qui apprendrait « beaucoup de mentions
très bien implique une formation sélective » sur ces sessions apprendrait le
mode d'attribution du diplôme, pas la sélectivité.

**Le statut de boursier, en 2019 puis en 2025.** La part de vœux issus de
boursiers passe de 12,5 % à 13,8 % puis à 16,3 % en 2020, s'y maintient quatre
ans, et **rechute à 13,8 % en 2025**. Deux marches, dans les deux sens.

Ce point est bien plus gênant que le précédent, pour une raison précise : **le
statut de boursier est une dimension de mes cellules de label**. Mes cellules
`_brs` de la session 2025 ne décrivent donc pas la même population que celles des
sessions d'entraînement. C'est une dérive de covariable qui tombe exactement sur
mon jeu de test.

Je ne peux pas l'empêcher. Je peux la mesurer, la déclarer **avant** de regarder
mes résultats, ventiler mes métriques par statut de boursier pour savoir d'où
vient un éventuel écart, et l'inscrire comme signal à surveiller en production.

## 4. La session 2020 est-elle utilisable ?

La question se pose sérieusement : si 2020 est une année aberrante, il ne me
reste que trois sessions d'entraînement. Mais l'anomalie constatée porte sur les
mentions, c'est-à-dire sur des variables explicatives. Encore faut-il vérifier ce
qu'il en est de **la cible elle-même**.

In [ ]:
print(f"{'année':<7}{'taux agrégé':>14}{'moy. bornée':>13}{'médiane':>10}{'part à 1':>11}{'part à 0':>11}{'tension méd.':>14}")
for annee in [a for a in MILLESIMES if a >= 2020]:
    d = charger(annee)
    x = d.filter(pl.col("nb_voe_pp_bg") > 0).with_columns(
        (pl.col("prop_tot_bg") / pl.col("nb_voe_pp_bg")).clip(0, 1).alias("taux")
    )
    voeux, propositions = d["nb_voe_pp_bg"].sum(), d["prop_tot_bg"].sum()
    tension = (
        d.filter((pl.col("capa_fin") > 0) & (pl.col("voe_tot") > 0))
        .with_columns((pl.col("voe_tot") / pl.col("capa_fin")).alias("t"))["t"]
        .median()
    )
    t = x["taux"]
    print(
        f"{annee:<7}{propositions / voeux * 100:>13.1f}%{t.mean():>13.3f}{t.median():>10.3f}"
        f"{(t == 1).mean() * 100:>10.1f}%{(t == 0).mean() * 100:>10.1f}%{tension:>14.1f}"
    )

**Ce que j'en conclus, et c'est une décision que j'assume.**

**La session 2020 est utilisable.** Sa cible est la plus basse de la série, mais
elle s'inscrit dans une progression régulière, sans décrochage. La tension
médiane, à 11,8 vœux par place, est parfaitement alignée sur les autres sessions.
Les masses aux bornes sont dans la norme.

Le constat mérite d'être formulé pour lui-même : **une session anormale sur
certaines variables peut être parfaitement normale sur la cible.** Écarter 2020
« parce que c'était l'année du confinement » aurait été un réflexe coûteux — un
quart de mes données d'entraînement — et non fondé sur une mesure.

### Le protocole d'évaluation que j'arrête

```
entraînement : 2020, 2021, 2022, 2023
validation   : 2024
test         : 2025
```

Assorti de trois réserves, écrites avant de voir le moindre résultat :

1. **Les variables de mention sont écartées ou signalées pour 2020 et 2021**,
   contaminées par le mode d'attribution du baccalauréat.
2. **Les métriques sont ventilées par statut de boursier**, dont la composition
   rompt entre les sessions d'entraînement et la session de test.
3. **Une dégradation est attendue entre validation et test**, la cible n'étant
   pas stationnaire. Elle ne devra pas être imputée d'emblée au modèle.

## Bilan de la phase exploratoire

| Question | Ce que j'ai établi |
|---|---|
| Cible calculable | seulement à partir de **2020** : le numérateur ventilé par baccalauréat n'existe pas avant |
| Volume réel | **440 030 cellules** sur six sessions, et non 560 000 à 625 000 sur huit |
| Stationnarité | non : taux agrégé en baisse, moyenne par formation en hausse — les deux définitions divergent |
| Rupture 2020 | contrôle continu : +4,4 points de mentions très bien, −13,6 points de sans-mention |
| Rupture boursiers | deux marches, en 2020 puis en **2025**, soit sur le jeu de test lui-même |
| Session 2020 | **conservée** : anomalie sur les mentions, pas sur la cible |

Cette phase exploratoire se referme sur une révision de mon protocole
d'évaluation, fondée sur des mesures et non sur des hypothèses. La configuration
du projet doit être mise à jour en conséquence, et chacune de ces décisions
consignée.